In [1]:
from datetime import datetime, timedelta
from utils.spark_session import get_sparkSession 
from utils.utilities import generate_date
from utils.dq_checks import dq_validate_checks

In [2]:
from pyspark.sql.functions import current_timestamp, current_date, lit, expr, to_date, when, lower, upper, trim, concat_ws, xxhash64, cast, col, lag, date_sub, coalesce, date_format
from pyspark.sql.window import Window

In [3]:
spark = get_sparkSession("Load gold.fact_inventory")

In [4]:
print("SPARK-APP : spark-UI - " + spark.sparkContext.uiWebUrl)

SPARK-APP : spark-UI - http://ba7b2bbb9aec:4040


In [5]:
from utils.load_controller import insert_control_logs, get_max_loadTimestamp
from delta import DeltaTable

In [6]:
_schema_name = "gold"
_table_name = "fact_inventory"
_fullname = f"{_schema_name}.{_table_name}"
_script_name = "gold_fact_inventory.ipynb"
_silver_table = "silver.fact_inventory"

print(f"SPARK-APP : Loading started for {_fullname}")

SPARK-APP : Loading started for gold.fact_inventory


In [7]:
#spark config

spark.conf.set("spark.sql.shuffle.partitions",16)


In [8]:
#Reading from silver.customer and generating surrogate key and business key

df = spark.read.table(_silver_table)

print(f"SPARK-APP: Silver Table Data count : {df.count()}")
df.show(truncate = False)

SPARK-APP: Silver Table Data count : 6
+--------------+----------+------+-------+--------+----------------+-----------------+------------------+----------+--------------------------+---------------------------+--------------------------+---------------------------+------------------+
|inventory_date|product_id|store |channel|currency|quantity_on_hand|quantity_reserved|quantity_available|cost_price|created_on                |created_by                 |modified_on               |modified_by                |source_file       |
+--------------+----------+------+-------+--------+----------------+-----------------+------------------+----------+--------------------------+---------------------------+--------------------------+---------------------------+------------------+
|2025-01-01    |P1001     |AMZ_US|MKT    |USD     |100             |10               |90                |350.00    |2026-01-29 01:05:34.293464|silver_fact_inventory.ipynb|2026-01-29 01:05:34.293464|silver_fact_inventory.ipy

In [9]:
# DQ validations

_row = df \
        .select("source_file") \
        .distinct() \
        .limit(1) \
        .first()

_source_file = "UNKNOWN" if _row is None else _row[0]

df = dq_validate_checks(df, spark, _schema_name, _table_name, _source_file)

print("SPARK-APP: DQ validations completed")
print(f"SPARK-APP: Table Data count after DQ validations : {df.count()}")

SPARK-APP: DQ validations completed
SPARK-APP: Table Data count after DQ validations : 6


In [10]:
#Adding audit columns

df = df.withColumn("created_on", current_timestamp()) \
       .withColumn("created_by", lit(_script_name)) \
       .withColumn("modified_on", current_timestamp()) \
       .withColumn("modified_by", lit(_script_name))

print(f"SPARK-APP: Silver Table Data count : {df.count()}")

# Generating surrogate key and business key

df_sk = df.withColumn("inventory_sk", xxhash64(concat_ws("||", "inventory_date", "product_id", "store", "channel")).cast("bigint"))

df_sk.show(truncate = False)

SPARK-APP: Silver Table Data count : 6
+--------------+----------+------+-------+--------+----------------+-----------------+------------------+----------+------------------------+-------------------------+------------------------+-------------------------+------------------+--------------------+
|inventory_date|product_id|store |channel|currency|quantity_on_hand|quantity_reserved|quantity_available|cost_price|created_on              |created_by               |modified_on             |modified_by              |source_file       |inventory_sk        |
+--------------+----------+------+-------+--------+----------------+-----------------+------------------+----------+------------------------+-------------------------+------------------------+-------------------------+------------------+--------------------+
|2025-01-01    |P1001     |AMZ_US|MKT    |USD     |100             |10               |90                |350.00    |2026-01-29 03:40:50.1243|gold_fact_inventory.ipynb|2026-01-29 03:40:

In [11]:
# Join with other dimension tables to get their surrogate keys

df_gold_store = spark.read.table("gold.dim_store").select("store_sk", "store_code")
df_gold_channel = spark.read.table("gold.dim_channel").select("channel_sk", "channel_code")
df_inventory_date = spark.read.table("gold.dim_date").select("date_sk","date")
df_gold_currency = spark.read.table("gold.dim_currency").select("currency_sk", "currency_code")

df_silver = df_sk.join(df_gold_store, how="inner", on=df_sk.store==df_gold_store.store_code) \
                 .join(df_gold_channel, how="inner", on=df_sk.channel==df_gold_channel.channel_code) \
                 .join(df_gold_currency, how = "inner", on=df_sk.currency==df_gold_currency.currency_code) \
                 .join(df_inventory_date, how = "left", on=df_sk.inventory_date==df_inventory_date.date) \
                 .select("inventory_sk", coalesce(df_inventory_date.date_sk,date_format(df_sk.inventory_date, "yyyyMMdd").cast("int")).alias("inventory_date"),                          
                         "product_id", "store_sk", "channel_sk", "currency_sk", "cost_price", "quantity_on_hand", "quantity_reserved", "quantity_available", 
                         "source_file", "created_on", "created_by", "modified_on", "modified_by")

df_silver.show(truncate = False)

+--------------------+--------------+----------+--------+----------+-----------+----------+----------------+-----------------+------------------+------------------+--------------------------+-------------------------+--------------------------+-------------------------+
|inventory_sk        |inventory_date|product_id|store_sk|channel_sk|currency_sk|cost_price|quantity_on_hand|quantity_reserved|quantity_available|source_file       |created_on                |created_by               |modified_on               |modified_by              |
+--------------------+--------------+----------+--------+----------+-----------+----------+----------------+-----------------+------------------+------------------+--------------------------+-------------------------+--------------------------+-------------------------+
|-3585030345938739901|20250102      |P1001     |2       |1         |1          |350.00    |98              |10               |88                |fact_inventory.csv|2026-01-29 03:41:06.456

In [12]:
#Derive additional columns

window = Window.partitionBy("product_id", "store_sk", "channel_sk").orderBy(col("inventory_date"))

df_silver = df_silver.withColumn("quantity_on_hand_yesterday", lag(col("quantity_on_hand")).over(window)) \
                     .withColumn("quantity_on_hand_change", when(col("quantity_on_hand_yesterday").isNull(), 0).otherwise(col("quantity_on_hand") - col("quantity_on_hand_yesterday"))) \
                     .drop("quantity_on_hand_yesterday")
                                 
df_silver.show()
df_silver.printSchema()

+--------------------+--------------+----------+--------+----------+-----------+----------+----------------+-----------------+------------------+------------------+--------------------+--------------------+--------------------+--------------------+-----------------------+
|        inventory_sk|inventory_date|product_id|store_sk|channel_sk|currency_sk|cost_price|quantity_on_hand|quantity_reserved|quantity_available|       source_file|          created_on|          created_by|         modified_on|         modified_by|quantity_on_hand_change|
+--------------------+--------------+----------+--------+----------+-----------+----------+----------------+-----------------+------------------+------------------+--------------------+--------------------+--------------------+--------------------+-----------------------+
| 9017425531003032706|      20250101|     P1001|       2|         1|          1|    350.00|             100|               10|                90|fact_inventory.csv|2026-01-29 03:43:

In [13]:
# Truncate table for full load
dt_fact_inventory = DeltaTable.forName(spark, f"{_schema_name}.{_table_name}")

if get_max_loadTimestamp(spark, _schema_name, _table_name) == '1900-01-01 00:00:00.000000':
    
    #Full-load so truncate dim table
    spark.sql("SET spark.databricks.delta.retentionDurationCheck.enabled=false")
    
    dt_fact_inventory.delete()
    dt_fact_inventory.vacuum(0)


In [14]:
# Merge
dt_fact_inventory.alias("t").merge(
    df_silver.alias("s"),
    "t.inventory_sk = s.inventory_sk"
).whenMatchedUpdate(set = {   
    "cost_price":"s.cost_price",
    "quantity_on_hand":"s.quantity_on_hand",
    "quantity_reserved":"s.quantity_reserved",
    "quantity_available":"s.quantity_available",
    "quantity_on_hand_change":"s.quantity_on_hand_change",
    "source_file" : "s.source_file",
    "modified_on" : "s.modified_on",
    "modified_by" : "s.modified_by"
    }).whenNotMatchedInsertAll().execute()
                     
print("SPARK-APP: Merge completed") 

SPARK-APP: Merge completed


In [15]:
hist = dt_fact_inventory.history().limit(1).select("version", "operationMetrics.executionTimeMs",
                                          "operationMetrics.numTargetRowsInserted", "operationMetrics.numTargetRowsUpdated",
                                          "operationMetrics.numOutputRows")

hist.show()
row = hist.first()

loaded_count = int("0" if row is None else row["numTargetRowsInserted"]) + int("0" if row is None else row["numTargetRowsUpdated"])


+-------+---------------+---------------------+--------------------+-------------+
|version|executionTimeMs|numTargetRowsInserted|numTargetRowsUpdated|numOutputRows|
+-------+---------------+---------------------+--------------------+-------------+
|      1|           6992|                    6|                   0|            6|
+-------+---------------+---------------------+--------------------+-------------+



In [16]:
spark.read.table(f"{_schema_name}.{_table_name}").show()

+--------------------+--------------+----------+--------+----------+-----------+----------+----------------+-----------------+------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+------------------+
|        inventory_sk|inventory_date|product_id|store_sk|channel_sk|currency_sk|cost_price|quantity_on_hand|quantity_reserved|quantity_available|quantity_on_hand_change|          created_on|          created_by|         modified_on|         modified_by|       source_file|
+--------------------+--------------+----------+--------+----------+-----------+----------+----------------+-----------------+------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+------------------+
|  545467236697362995|      20250102|     P1003|       5|         1|          1|    350.00|             100|               10|                90|                      0|2026-01-29 0

In [17]:
#Writing log data to load_controller

_data = []

_data.append([_schema_name, _schema_name, _table_name, "delta-load", "merge", str(datetime.now()), "success", loaded_count, _script_name, _script_name])

insert_control_logs(spark, _data)
    
print(f"SPARK-APP: Data written to load_controller")

SPARK-APP: Data written to load_controller


In [18]:
spark.sql(f"select * from gold.load_controller where schema_name = '{_schema_name}' and table_name = '{_table_name}' order by created_on desc").show()

+-----+-----------+--------------+----------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
|layer|schema_name|    table_name| load_type|write_mode|      load_timestamp|load_status|loaded_count|          created_on|          created_by|         modified_on|         modified_by|
+-----+-----------+--------------+----------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
| gold|       gold|fact_inventory|delta-load|     merge|2026-01-29 03:44:...|    success|           6|2026-01-29 03:44:...|gold_fact_invento...|2026-01-29 03:44:...|gold_fact_invento...|
+-----+-----------+--------------+----------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+



In [19]:
#Generating symlink manifest file

dt = DeltaTable.forName(spark, f"{_schema_name}.{_table_name}")
dt.generate("symlink_format_manifest")

print("SPARK-APP: symlink manifest file generated")

SPARK-APP: symlink manifest file generated


In [20]:
spark.stop()